# 영상 채널 성과 분석 (2019–2021) — 분석 노트북

직접 운영한 영상 채널의 YouTube Studio 데이터를 사후 분석한 노트북입니다. 사이트(`index.html`)의 모든 수치는 이 노트북의 출력과 같습니다.

**역할과 시점**
- 2019–2021: 채널 운영(콘텐츠 기획·편집·게시)
- 2026: 이 분석(데이터 추출, 정합성 점검, Python 분석, 리포트 작성). 유료 광고 집행 경험이 아닙니다.

**데이터 출처:** YouTube Studio 분석 → 고급 모드 → CSV 내보내기 (본인 채널)

| 원본 파일 (`data/raw/`) | 내보낸 화면 · 기간 | 사용한 열 | 추출일 |
|---|---|---|---|
| `videos_2019_2021.csv` | 콘텐츠 탭 표 · 2019-01-13~2021-12-31 | 노출수, CTR, 조회수, 평균 시청 지속 시간, 평균 조회율, 시청 시간, 구독자, 길이, 게시일 | 2026-11-02 |
| `traffic_2019_2021.csv` | 트래픽 소스 탭 표 · 같은 기간 | 노출수, CTR, 조회수, 평균 시청 지속 시간, 평균 조회율, 시청 시간 | 2026-11-02 |
| `traffic_daily_2019_2021.csv` | 트래픽 소스 탭 일별 차트 · 같은 기간 | 날짜, 트래픽 소스, 조회수 | 2026-11-02 |
| `channel_daily_views_lifetime.csv` | 콘텐츠 탭 일별 총계 · 2019-01-13~2026-10-31 | 날짜, 조회수 | 2026-11-02 |
| `subscribers_daily_lifetime.csv` | 구독자 일별 총계 · 2019-01-13~2026-10-31 | 날짜, 구독자(순증가) | 2026-11-02 |
| `top5_video_daily_lifetime.csv` | 콘텐츠 탭 일별 차트(조회수 상위 5개 영상) · 2019-01-13~2026-10-31 | 날짜, 길이, 조회수 | 2026-11-02 |

원본 CSV에는 영상 제목·ID가 포함되어 있어 공개하지 않습니다. 따라서 **공개 저장소만으로는 이 노트북을 재실행할 수 없고**, 익명화된 결과(`data/processed/`)와 아래 출력으로 결과를 확인할 수 있습니다.

**표기 원칙**
- **[관측]** 원본에 있는 값 그대로
- **[계산]** 원본 값의 합·차·비율 (분자·분모 명시)
- **[참고]** 가정을 둔 계산값 (가정 명시, 결론의 근거로 쓰지 않음)
- **[추정]** 근거는 있으나 직접 확인할 수 없는 해석
- **[가설]** 검증하지 않은 해석

In [1]:
import pandas as pd
RAW, OUT = 'data/raw', 'data/processed'
START, END = '2019-01-13', '2021-12-31'
SPIKE = ('2019-03-09', '2019-03-22')   # 급증 14일
pd.set_option('display.width', 140)
def hms(s):
    h, m, sec = map(int, str(s).split(':')); return h*3600 + m*60 + sec

## 1. 데이터 준비와 익명화

영상은 제목·ID 대신 **영상 길이(초)**로 매칭해 유형 코드를 붙입니다. 공개 상태 이력은 **운영자 기록**(비공개 처리 후 재공개한 영상 목록)이며, 비공개 시작 시점은 일별 조회수로 **추정**한 값입니다(2절).

In [2]:
VIDEO_MAP = {  # 길이(초): (코드, 유형, 공개 상태 이력 — 운영자 기록)
    2131: ('L1', '롱폼 음성 콘텐츠', '비공개 이력 없음'),
    1353: ('L2', '롱폼 음성 콘텐츠', '비공개 이력 있음 → 2026 재공개'),
    2260: ('L3', '롱폼 음성 콘텐츠', '비공개 이력 있음 → 2026 재공개'),
    231:  ('S1', '콘셉트 숏폼(영상통화 형식)', '비공개 이력 없음'),
    64:   ('C1', '예능 클립', '비공개 이력 있음 → 2026 재공개'),
    51:   ('C2', '예능 클립', '비공개 이력 있음 → 2026 재공개'),
    374:  ('M1', '편집 영상', '비공개 이력 있음 → 2026 재공개'),
    141:  ('M2', '편집 영상', '비공개 이력 있음 → 2026 재공개'),
    3111: ('P1', '플레이리스트', '저작권 부분 차단 → 비공개'),
}
v = pd.read_csv(f'{RAW}/videos_2019_2021.csv')
total = v[v['콘텐츠'] == '합계'].iloc[0]
v = v[v['콘텐츠'] != '합계'].copy()
v[['code', 'type', 'status']] = v['길이'].map(VIDEO_MAP).apply(pd.Series)
v = v.rename(columns={'길이': 'length_s', '유효 조회수': 'views', '시청 시간(단위: 시간)': 'watch_h',
                      '구독자': 'subs', '썸네일 노출수': 'impr', '썸네일 클릭률 (%)': 'ctr',
                      '평균 조회율 (%)': 'pct_viewed'})
v['avg_view_s'] = v['평균 시청 지속 시간'].map(hms)
v['published'] = pd.to_datetime(v['동영상 게시 시간'], format='%b %d, %Y').dt.date
v = v[['code', 'type', 'status', 'published', 'length_s', 'impr', 'ctr', 'views', 'avg_view_s', 'pct_viewed', 'watch_h', 'subs']]
v.sort_values('views', ascending=False).reset_index(drop=True)

,code,type,status,published,length_s,impr,ctr,views,avg_view_s,pct_viewed,watch_h,subs
0,L1,롱폼 음성 콘텐츠,비공개 이력 없음,2019-01-19,2131.0,127652,17.04,25600,226,10.63,1610.7941,31
1,S1,콘셉트 숏폼(영상통화 형식),비공개 이력 없음,2019-07-13,231.0,65837,19.12,14863,59,25.71,245.2125,29
2,L2,롱폼 음성 콘텐츠,비공개 이력 있음 → 2026 재공개,2019-07-30,1353.0,44764,8.72,4836,210,15.53,282.2400,13
3,L3,롱폼 음성 콘텐츠,비공개 이력 있음 → 2026 재공개,2019-07-28,2260.0,29628,11.90,4114,273,12.10,312.4636,3
4,C1,예능 클립,비공개 이력 있음 → 2026 재공개,2019-08-11,64.0,12869,6.39,1005,50,79.11,14.1352,1
5,M2,편집 영상,비공개 이력 있음 → 2026 재공개,2020-09-05,141.0,1972,11.46,266,40,28.50,2.9688,0
6,C2,예능 클립,비공개 이력 있음 → 2026 재공개,2019-08-25,51.0,5352,3.59,249,32,63.21,2.2296,1
7,M1,편집 영상,비공개 이력 있음 → 2026 재공개,2019-07-22,374.0,4092,3.32,205,62,16.72,3.5614,0
8,P1,플레이리스트,저작권 부분 차단 → 비공개,2021-07-23,3111.0,238,5.04,30,52,1.70,0.4400,-1


**제외한 지표:** 순 시청자수·신규/재방문 시청자·고정 시청자·공동 시청은 이 기간 값이 비어 있거나 0이라 사용하지 않습니다. 따라서 이 분석의 '조회수'는 반복 시청이 포함될 수 있는 **조회 규모**이며, 중복을 제거한 도달 인원이 아닙니다.

## 2. 분석 기간과 공개 상태 점검

In [3]:
daily = pd.read_csv(f'{RAW}/channel_daily_views_lifetime.csv', parse_dates=['날짜'])
daily.columns = ['date', 'views']; daily = daily.set_index('date').sort_index()['views']
life, core = daily.sum(), daily[START:END].sum()
print('[계산] 분석 기간 조회수 / 전체 기간 조회수 = %d / %d = %.1f%%  (전체 기간: %s ~ %s)' % (core, life, core / life * 100, daily.index.min().date(), daily.index.max().date()))

top5 = pd.read_csv(f'{RAW}/top5_video_daily_lifetime.csv', parse_dates=['날짜'])
print()
print('일별 조회수 0이 이어진 구간 (첫 조회 이후)')
for L in [1353, 2260, 64]:
    g = top5[top5['길이'] == L].set_index('날짜')['유효 조회수'].sort_index()
    g = g[g[g > 0].index[0]:]
    z = (g == 0); runs = z.groupby((z != z.shift()).cumsum())
    core_max = int(z[:END].groupby((z[:END] != z[:END].shift()).cumsum()).sum().max())
    last = g[g > 0].index[-1].date()
    print('  %s: 2019–2021 최장 연속 0일 %d일 / 마지막 조회일 %s 이후 0' % (VIDEO_MAP[L][0], core_max, last))

[계산] 분석 기간 조회수 / 전체 기간 조회수 = 123577 / 129374 = 95.5%  (전체 기간: 2019-01-13 ~ 2026-10-31)

일별 조회수 0이 이어진 구간 (첫 조회 이후)
  L2: 2019–2021 최장 연속 0일 6일 / 마지막 조회일 2025-07-01 이후 0
  L3: 2019–2021 최장 연속 0일 5일 / 마지막 조회일 2025-06-23 이후 0
  C1: 2019–2021 최장 연속 0일 19일 / 마지막 조회일 2023-10-01 이후 0


- **[계산]** 분석 기간(2019-01-13~2021-12-31)에 전체 기간(~2026-10-31) 조회수의 95.5%가 발생했습니다.
- **[추정]** L2·L3·C1의 비공개 시작 시점은 '마지막 조회일 이후 조회수 0'으로 추정한 값입니다(L2 2025-07, L3 2025-06, C1 2023-10).
- **[관측]** 2019–2021년 L2·L3는 조회수 0이 최장 5–6일만 이어졌습니다. C1은 조회 규모가 작아 19일 연속 0인 구간이 있어 판단이 어렵습니다.
- 따라서 '공개 상태 차이를 통제했다'가 아니라 **'확인 가능한 범위에서 영향을 줄였다'**로 봅니다. C2·M1·M2·P1은 일별 자료가 없어 비공개 시점을 확인하지 못했습니다.

## 3. 집계 범위 점검: 채널 합계 vs 현존 영상 합계

YouTube는 삭제된 콘텐츠의 기록도 채널 합계에 남깁니다. 조회수·시청 시간·노출수는 영상별 값을 더하면 채널 합계가 되는 지표이므로, 차이를 **현존 영상 목록에서 확인되지 않는 성과(잔여분 R)**로 볼 수 있습니다(현존 영상 목록이 완전하다는 전제).

구독자는 다릅니다. 채널 구독은 영상 시청 페이지 외에도 채널 페이지 등에서 발생할 수 있어, **채널 합계와 영상별 합계의 집계 범위가 같다고 볼 수 없습니다.** 구독 차이는 잔여분으로 귀속하지 않습니다.

In [4]:
ch = dict(views=int(total['유효 조회수']), watch_h=float(total['시청 시간(단위: 시간)']),
          impr=int(total['썸네일 노출수']), subs=int(total['구독자']))
ex = dict(views=int(v.views.sum()), watch_h=float(v.watch_h.sum()), impr=int(v.impr.sum()), subs=int(v.subs.sum()))
R = {k: ch[k] - ex[k] for k in ['views', 'watch_h', 'impr']}
print('[관측] 채널 합계      :', ch)
print('[계산] 현존 9개 합계  :', {k: round(x, 4) for k, x in ex.items()})
print('[계산] 잔여분 R       :', {k: round(x, 4) for k, x in R.items()})
print('[계산] 현존 영상 조회수 비중 = %d / %d = %.1f%%' % (ex['views'], ch['views'], ex['views'] / ch['views'] * 100))
print('[계산] 잔여분 비중: 조회 %.1f%% · 시청 시간 %.1f%% · 노출 %.1f%%' % tuple(R[k] / ch[k] * 100 for k in ['views', 'watch_h', 'impr']))
print('[계산] 구독 집계 차이 = 채널 %+d − 영상별 합계 %+d = %+d명 (귀속 불가)' % (ch['subs'], ex['subs'], ch['subs'] - ex['subs']))

[관측] 채널 합계      : {'views': 123577, 'watch_h': 5749.6093, 'impr': 2868407, 'subs': 218}
[계산] 현존 9개 합계  : {'views': 51168, 'watch_h': 2474.0452, 'impr': 292404, 'subs': 77}
[계산] 잔여분 R       : {'views': 72409, 'watch_h': 3275.5641, 'impr': 2576003}
[계산] 현존 영상 조회수 비중 = 51168 / 123577 = 41.4%
[계산] 잔여분 비중: 조회 58.6% · 시청 시간 57.0% · 노출 89.8%
[계산] 구독 집계 차이 = 채널 +218 − 영상별 합계 +77 = +141명 (귀속 불가)


## 4. 일별 모니터링: 급증과 급감

In [5]:
print(daily['2019-03-05':'2019-03-26'].to_frame().T)
spike_ch = int(daily[SPIKE[0]:SPIKE[1]].sum())
print('[계산] 급증 14일 조회수 / 분석 기간 조회수 = %d / %d = %.1f%%' % (spike_ch, core, spike_ch / core * 100))

print('2019-03-09 이전 게시된 현존 영상:', v.loc[pd.to_datetime(v.published) < '2019-03-09', 'code'].tolist())
l1 = top5[top5['길이'] == 2131].set_index('날짜')['유효 조회수'].sort_index()
l1_spike = int(l1[SPIKE[0]:SPIKE[1]].sum())
print('[관측] L1 일별 조회수 — 급증 14일: %d–%d회, 직전 4주 평균 %.0f회' % (l1[SPIKE[0]:SPIKE[1]].min(), l1[SPIKE[0]:SPIKE[1]].max(), l1['2019-02-09':'2019-03-08'].mean()))
r_spike = spike_ch - l1_spike
print('[계산] 급증 14일 중 현존 영상 외 조회수 = %d − L1 %d = %d회' % (spike_ch, l1_spike, r_spike))
print('[계산] 잔여분 R 조회수 중 급증 14일 비중 = %d / %d = %.1f%%' % (r_spike, R['views'], r_spike / R['views'] * 100))

t5_sum = top5.groupby('날짜')['유효 조회수'].sum().reindex(daily.index).fillna(0)
resid = daily - t5_sum
print('[계산] 급증 이후(2019-03-23~07-21, 당시 현존 영상은 L1뿐) 현존 영상 외 조회수: %d회' % resid['2019-03-23':'2019-07-21'].sum())

date   2019-03-05  2019-03-06  2019-03-07  2019-03-08  2019-03-09  2019-03-10  2019-03-11  2019-03-12  2019-03-13  2019-03-14  ...  \
views          66          59          90         134        1091         471         386        1126        2649        3506  ...   

date   2019-03-17  2019-03-18  2019-03-19  2019-03-20  2019-03-21  2019-03-22  2019-03-23  2019-03-24  2019-03-25  2019-03-26  
views        6195        6224        6482        6195        7293       14278          46          48          24          27  

[1 rows x 22 columns]
[계산] 급증 14일 조회수 / 분석 기간 조회수 = 70636 / 123577 = 57.2%
2019-03-09 이전 게시된 현존 영상: ['L1']
[관측] L1 일별 조회수 — 급증 14일: 18–68회, 직전 4주 평균 48회
[계산] 급증 14일 중 현존 영상 외 조회수 = 70636 − L1 508 = 70128회
[계산] 잔여분 R 조회수 중 급증 14일 비중 = 70128 / 72409 = 96.8%
[계산] 급증 이후(2019-03-23~07-21, 당시 현존 영상은 L1뿐) 현존 영상 외 조회수: 1181회


- **[관측]** 2019년 3월 9–22일 일별 조회수가 평소(하루 수십 회)의 100배 이상으로 늘었고, 3월 22일 14,278회에서 다음 날 46회로 급감했습니다.
- **[계산]** 급증 14일 조회수는 분석 기간 조회수의 57.2%입니다.
- **[계산]** 급증 당시 게시돼 있던 현존 영상은 L1 하나이며, L1은 평소 수준이었습니다. 급증 조회수의 대부분(70,128회)은 **현존 영상 목록에서 확인되지 않는 콘텐츠**에서 발생했고, 이는 잔여분 R 조회수의 96.8%입니다.
- **[추정]** 급증 콘텐츠는 운영자 기록상 저작권 문제로 삭제된 예능 하이라이트 편집 영상(4분 32초)일 가능성이 높습니다. 삭제 날짜 기록은 없어, 3월 22일 급감이 삭제 때문인지는 확정하지 않습니다.
- **[참고]** 잔여분의 평균 시청 시간(시청 시간 R ÷ 조회수 R ≈ 2분 43초)은 4분 32초보다 짧아 위 추정과 모순되지 않습니다. 다만 이는 추정을 뒷받침하는 독립 증거가 아닙니다.
- **[관측]** 급증 이후에도 현존 영상으로 설명되지 않는 조회수가 1,181회 있어, 삭제 콘텐츠가 1개보다 많았거나 3월 22일 급감이 삭제가 아닌 다른 제한이었을 가능성이 있습니다.

## 5. 유입 구조: 급증 14일 vs 나머지 기간

In [6]:
t = pd.read_csv(f'{RAW}/traffic_daily_2019_2021.csv', parse_dates=['날짜'])
t.columns = ['date', 'source', 'views']
t = t[(t.date >= START) & (t.date <= END)]
GROUP = {'추천 동영상': '추천·탐색', '탐색 기능': '추천·탐색', 'YouTube 검색': '검색'}
t['group'] = t.source.map(GROUP).fillna('기타')
in_spike = t.date.between(*SPIKE)
split = (t.assign(period=in_spike.map({True: '급증 14일', False: '나머지 기간'}))
          .pivot_table(index='period', columns='group', values='views', aggfunc='sum'))
split['합계'] = split.sum(axis=1)
pct = split[['추천·탐색', '검색', '기타']].div(split['합계'], axis=0).mul(100).round(1)
print(split); print(pct)
rec = t[t.source == '추천 동영상']
print('[계산] 추천 동영상 유입 중 급증 14일 비중 = %d / %d = %.2f%%' % (rec[rec.date.between(*SPIKE)].views.sum(), rec.views.sum(), rec[rec.date.between(*SPIKE)].views.sum() / rec.views.sum() * 100))
m = t.assign(month=t.date.dt.to_period('M').astype(str)).pivot_table(index='month', columns='group', values='views', aggfunc='sum').fillna(0).astype(int)
print('[관측] 월별 검색 유입: 2020년 연간 %d회 → 2021년 연간 %d회' % (m.loc[m.index.str.startswith('2020'), '검색'].sum(), m.loc[m.index.str.startswith('2021'), '검색'].sum()))

group      검색    기타  추천·탐색     합계
period                           
급증 14일   1170  1007  68459  70636
나머지 기간  44155  5347   3439  52941
group   추천·탐색    검색    기타
period                   
급증 14일   96.9   1.7   1.4
나머지 기간    6.5  83.4  10.1
[계산] 추천 동영상 유입 중 급증 14일 비중 = 56823 / 57553 = 98.73%
[관측] 월별 검색 유입: 2020년 연간 19814회 → 2021년 연간 9641회


- **[계산]** 급증 14일 조회수(70,636회)의 96.9%는 추천·탐색에서, 나머지 기간 조회수(52,941회)의 83.4%는 검색에서 발생했습니다. 추천 동영상 유입 전체의 98.73%가 급증 14일에 몰려 있습니다.
- **[관측]** 검색 유입은 관찰 기간 내내 매달 발생했고, 2020년 이후 전반적으로 감소했습니다. 이는 관찰 기간의 지속성이며, 이후에도 유지되는 수요라는 뜻은 아닙니다. 감소 원인(업로드 빈도 등)은 확인하지 않았습니다.

## 6. 유입 경로별 지표

In [7]:
ts = pd.read_csv(f'{RAW}/traffic_2019_2021.csv')
ts = ts.rename(columns={'트래픽 소스': 'source', '유효 조회수': 'views', '평균 조회율 (%)': 'pct_viewed',
                        '시청 시간(단위: 시간)': 'watch_h', '썸네일 노출수': 'impr', '썸네일 클릭률 (%)': 'ctr'})
ts['avg_view_s'] = ts['평균 시청 지속 시간'].map(hms)
ts = ts[['source', 'impr', 'ctr', 'views', 'avg_view_s', 'pct_viewed', 'watch_h']]
tot = ts[ts.source == '합계'].iloc[0]
ts['impr_share'] = ts.impr / tot.impr * 100
ts['view_share'] = ts.views / tot.views * 100
main = ts[ts.source.isin(['추천 동영상', 'YouTube 검색', '탐색 기능'])].round(2)
print('[참고] 노출수 × CTR = %.0f회 vs 전체 조회수 %d회 — 모든 조회가 집계 대상 썸네일 노출에서 발생하지는 않음' % (tot.impr * tot.ctr / 100, tot.views))
main

[참고] 노출수 × CTR = 98100회 vs 전체 조회수 123577회 — 모든 조회가 집계 대상 썸네일 노출에서 발생하지는 않음


,source,impr,ctr,views,avg_view_s,pct_viewed,watch_h,impr_share,view_share
1,추천 동영상,2417384.0,1.84,57553,165,57.62,2638.12,84.28,46.57
2,YouTube 검색,263110.0,15.39,45325,143,10.44,1802.90,9.17,36.68
3,탐색 기능,144281.0,6.86,14345,185,37.16,740.79,5.03,11.61


- **[관측]** 추천은 노출 비중 84.3%·CTR 1.84%, 검색은 노출 비중 9.2%·CTR 15.39%로, 노출 규모와 CTR이 크게 달랐습니다.
- **[관측]** 평균 조회율도 추천 57.6%, 검색 10.4%로 달랐습니다.
- **[해석]** 경로별 평균 조회율 차이에는 유입 이용자의 특성뿐 아니라 각 경로로 소비된 영상의 길이·콘텐츠 구성 차이가 함께 영향을 미쳤을 수 있습니다. 같은 시간을 시청했다면 긴 영상일수록 조회율은 낮게 나옵니다. 영상별 유입 경로 자료가 없어 이 효과들을 분리할 수 없으므로, **현재 집계만으로 경로 자체의 시청 품질을 판단하지 않습니다.** CTR 역시 콘텐츠 구성·노출 대상·이용 의도의 영향을 받으므로, 경로 간 CTR 차이를 경로의 우열로 해석하지 않습니다.

## 7. 영상별 시청 성과와 KPI 선택

- 시청 성과 비교: 조회 30회인 P1만 제외(평균값이 극소수 조회에 좌우됨) → 8개 영상
- CTR 비교: 노출 5,000회 이상 영상만 해석 (분석자가 소량 데이터의 영향을 줄이려고 정한 **탐색 기준**이며 통계적 신뢰도 기준이 아님)
- 영상마다 게시 시점이 달라 같은 달력 기간이라도 성과를 쌓은 시간이 다릅니다. 따라서 누적값은 **'해당 기간의 채널 성과 기여도'**이며, 영상 자체의 성과를 공정하게 비교한 값이 아닙니다.

In [8]:
cmp = v[v.code != 'P1'].copy()
cmp['ctr_note'] = cmp.impr.map(lambda x: '' if x >= 5000 else '노출 5,000회 미만(참고)')
cmp['watch_share'] = cmp.watch_h / ch['watch_h'] * 100
cmp = cmp[['code', 'type', 'published', 'length_s', 'impr', 'ctr', 'ctr_note', 'views', 'avg_view_s', 'pct_viewed', 'watch_h', 'watch_share', 'subs']].sort_values('length_s', ascending=False).round(2)
cmp

,code,type,published,length_s,impr,ctr,ctr_note,views,avg_view_s,pct_viewed,watch_h,watch_share,subs
5,L3,롱폼 음성 콘텐츠,2019-07-28,2260.0,29628,11.90,,4114,273,12.10,312.46,5.43,3
2,L1,롱폼 음성 콘텐츠,2019-01-19,2131.0,127652,17.04,,25600,226,10.63,1610.79,28.02,31
4,L2,롱폼 음성 콘텐츠,2019-07-30,1353.0,44764,8.72,,4836,210,15.53,282.24,4.91,13
7,M1,편집 영상,2019-07-22,374.0,4092,3.32,"노출 5,000회 미만(참고)",205,62,16.72,3.56,0.06,0
9,S1,콘셉트 숏폼(영상통화 형식),2019-07-13,231.0,65837,19.12,,14863,59,25.71,245.21,4.26,29
1,M2,편집 영상,2020-09-05,141.0,1972,11.46,"노출 5,000회 미만(참고)",266,40,28.50,2.97,0.05,0
3,C1,예능 클립,2019-08-11,64.0,12869,6.39,,1005,50,79.11,14.14,0.25,1
6,C2,예능 클립,2019-08-25,51.0,5352,3.59,,249,32,63.21,2.23,0.04,1


In [9]:
long = v[v.code.str.startswith('L')]; short = v[(v.code != 'P1') & (v.length_s < 300)]
print('[계산] 롱폼 L1·L2·L3 시청 시간 = %.4f시간 / 채널 %.4f시간 = %.1f%%' % (long.watch_h.sum(), ch['watch_h'], long.watch_h.sum() / ch['watch_h'] * 100))
print('[관측] 롱폼: 평균 조회율 %.1f–%.1f%% / 평균 시청 %d–%d초' % (long.pct_viewed.min(), long.pct_viewed.max(), long.avg_view_s.min(), long.avg_view_s.max()))
print('[관측] 5분 미만 영상(%s): 평균 조회율 %.1f–%.1f%% / 평균 시청 %d–%d초' % (','.join(short.code), short.pct_viewed.min(), short.pct_viewed.max(), short.avg_view_s.min(), short.avg_view_s.max()))

[계산] 롱폼 L1·L2·L3 시청 시간 = 2205.4977시간 / 채널 5749.6093시간 = 38.4%
[관측] 롱폼: 평균 조회율 10.6–15.5% / 평균 시청 210–273초
[관측] 5분 미만 영상(M2,C1,C2,S1): 평균 조회율 25.7–79.1% / 평균 시청 32–59초


- **[관측]** 이번 비교 대상에서는 짧은 영상이 평균 조회율에서, 긴 음성 콘텐츠가 평균 시청 시간에서 높은 값을 보였습니다. 롱폼 3개는 분석 기간 채널 시청 시간의 38.4%를 차지했습니다.
- **[해석]** 지표별 결과가 달랐으므로 콘텐츠 목적에 맞는 KPI 선택이 필요합니다. 유형별 사례 수가 적고 길이·주제·게시 시점이 함께 달라, 유형의 효과를 입증한 것은 아닙니다.
- **[가설]** CTR이 가장 높은 L1(17.04%)과 S1(19.12%)은 검색 유입 비중이 높았기 때문일 수 있습니다. 영상별 유입 경로를 확보하지 않아 검증하지 못했습니다.

## 8. 구독 지표와 집계 범위

구독자 수는 **순증가(신규 − 취소)**이며, 신규 구독 전환율이 아닙니다.

In [10]:
s = pd.read_csv(f'{RAW}/subscribers_daily_lifetime.csv', parse_dates=['날짜']); s.columns = ['date', 'subs']
s = s.set_index('date').sort_index()['subs'][START:END]
sp = int(s[SPIKE[0]:SPIKE[1]].sum())
print('[관측] 채널 일별 순증가 합계 (분석 기간): %+d명' % s.sum())
print('[계산] 급증 14일: %+d명 / 나머지 기간: %+d명' % (sp, s.sum() - sp))
print('[계산] 영상별 구독 지표 합계 (현존 9개, 분석 기간 전체): %+d명' % ex['subs'])
print('[계산] 채널 합계와 영상별 합계의 차이: %+d명 — 집계 범위가 달라 특정 영상에 귀속하지 않음' % (ch['subs'] - ex['subs']))

[관측] 채널 일별 순증가 합계 (분석 기간): +218명
[계산] 급증 14일: +205명 / 나머지 기간: +13명
[계산] 영상별 구독 지표 합계 (현존 9개, 분석 기간 전체): +77명
[계산] 채널 합계와 영상별 합계의 차이: +141명 — 집계 범위가 달라 특정 영상에 귀속하지 않음


- **[관측]** 분석 기간 채널 구독 순증가는 +218명이며, 일별 기준으로 급증 14일에 +205명, 나머지 기간에 +13명이 집계됐습니다. 이는 기간별 순증가의 기술적 비교이며, 같은 시청자 집단의 전환율이나 인과 효과가 아닙니다.
- **[관측]** 영상별 구독 지표 합계는 +77명입니다(분석 기간 전체 기준). 채널 합계와의 차이(+141명)는 채널 페이지 등 영상 밖 구독, 삭제 콘텐츠, 구독 취소 위치 등이 섞인 집계 범위 차이로, 특정 영상의 성과로 확정하지 않습니다.
- **[가설]** 나머지 기간의 채널 순증가가 작은 것은 구독 취소의 영향일 수 있으나, 신규·취소 내역을 분리한 자료가 없어 확인하지 못했습니다.

## 9. 데이터 한계와 재현 범위
- 영상 수가 적어 통계 검정은 하지 않았습니다. 모든 결론은 사례 관찰입니다.
- 잔여분 R은 현존 영상 목록이 완전하다는 전제의 계산값입니다. 삭제 콘텐츠가 몇 개였는지, 언제 삭제됐는지는 기록이 없습니다.
- 비공개 시점은 L2·L3·C1만 일별 데이터로 추정했고, 나머지는 확인하지 못했습니다.
- 영상별 유입 경로, 신규·취소 구독, 구독 발생 위치 자료가 없어 경로별 시청 품질과 구독 기여도는 판단하지 않았습니다.
- 원본 CSV는 비공개라 공개 저장소만으로는 재실행할 수 없습니다. 공개된 것은 익명화 결과(`data/processed/`)와 이 노트북의 출력입니다.
- 분석은 채널 운영이 끝난 뒤(2026년)에 수행했습니다.

In [11]:
# 익명화된 결과 저장 (제목·ID 없음)
v.to_csv(f'{OUT}/videos_2019_2021_anon.csv', index=False)
main.to_csv(f'{OUT}/traffic_sources_2019_2021.csv', index=False)
pct.to_csv(f'{OUT}/traffic_mix_spike_vs_rest.csv')
m.to_csv(f'{OUT}/monthly_views_by_source.csv')
pd.Series({'channel_views': ch['views'], 'existing_views': ex['views'], 'residual_views': R['views'],
           'channel_watch_h': ch['watch_h'], 'existing_watch_h': round(ex['watch_h'], 4), 'residual_watch_h': round(R['watch_h'], 4),
           'channel_impr': ch['impr'], 'existing_impr': ex['impr'], 'residual_impr': R['impr'],
           'residual_views_in_spike': r_spike, 'channel_subs': ch['subs'], 'existing_subs': ex['subs'],
           'subs_scope_gap_unattributed': ch['subs'] - ex['subs']}).to_csv(f'{OUT}/aggregation_check.csv', header=['value'])
sorted(__import__('os').listdir(OUT))

['aggregation_check.csv',
 'monthly_views_by_source.csv',
 'traffic_mix_spike_vs_rest.csv',
 'traffic_sources_2019_2021.csv',
 'videos_2019_2021_anon.csv']